# P16 Counterfactual context consistency check

本 notebook 用於期刊補充實驗：固定同一套 PO-D 測試題、同一個 partial outfit、同一個目標單品與同一個同類候選池，只修改一項既有情境因子，觀察描述與推薦排序是否產生可解釋變化。

本研究先前已將生成描述拆分為三個因素：weather、occasion、style。因此本 notebook 只針對這三個因素做反事實替換，不新增 activity/MET 作為反事實條件，以避免偏離既有實驗架構。


In [ ]:
# ------------------------------------------------------------
# 0. 路徑、參數與共用工具
# ------------------------------------------------------------
from pathlib import Path
from copy import deepcopy
from contextlib import nullcontext
import base64
import csv
import html
import json
import os
import pickle
import re
import sys
import textwrap

import numpy as np
import pandas as pd

VALID_ROOT = Path("/home/ester/valid_description")
MODEL_ROOT = VALID_ROOT / "text-conditioned-outfit-recommendation"
POLYVORE_ROOT = VALID_ROOT / "polyvore_data" / "polyvore_outfits"
FEATURE_ROOT = VALID_ROOT / "fashionclip_data"
OUT_ROOT = VALID_ROOT / "experiment data" / "journal_figures_tables_20260629"
DERIVED_DIR = OUT_ROOT / "derived_audits"
FIG_DIR = OUT_ROOT / "figures"

FRAG_JSONL = VALID_ROOT / "wos_split_results_v5_merged_retry_round3.jsonl"
DETAIL_CIR_CSV = MODEL_ROOT / "experiments" / "detail_cir_new_seed1.csv"

PAIR_CSV = DERIVED_DIR / "A44_counterfactual_context_pairs.csv"
RETRIEVAL_CSV = DERIVED_DIR / "A45_counterfactual_context_retrieval_seed1.csv"
REVIEW_CSV = DERIVED_DIR / "A46_counterfactual_context_manual_review_sheet.csv"
NOTE_TXT = DERIVED_DIR / "A47_counterfactual_context_experiment_note.txt"
FIG_SVG_A = FIG_DIR / "F34a_counterfactual_context_larger_response_example.svg"
FIG_SVG_B = FIG_DIR / "F34b_counterfactual_context_limited_response_example.svg"

SEED = 1
RANDOM_SEED = 42
N_PER_DIRECTION = 4
MIN_OFFICIAL_POOL = 3000
RUN_MODEL = True

BLUE = "#1f77b4"      # context-aware baseline
ORANGE = "#f28e2b"    # counterfactual context
RED = "#d62728"       # target item

DERIVED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

TEMP_RE = re.compile(r"(-?\d+(?:\.\d+)?)\s*(?:°\s*)?[cC]\b")
FORMAL_RE = re.compile(r"\b(formal|office|business|work|professional|interview|meeting|wedding|party|cocktail|elegant|tailored|blazer|suit)\b", re.I)
CASUAL_RE = re.compile(r"\b(casual|weekend|streetwear|street|relaxed|everyday|daily|chill|laid-back|lounge|vacation|travel)\b", re.I)


def load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def save_csv(path: Path, df: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")


def clean_text(value) -> str:
    if isinstance(value, list):
        value = " ".join(str(x) for x in value if str(x).strip())
    return re.sub(r"\s+", " ", str(value or "")).strip()


def parse_temperature(text: str) -> float:
    m = TEMP_RE.search(str(text))
    return float(m.group(1)) if m else np.nan


def read_categories(path: Path) -> dict:
    out = {}
    with path.open("r", encoding="utf-8") as f:
        for row in csv.reader(f):
            if len(row) >= 3:
                out[str(row[0])] = {
                    "fine_category": row[1].strip(),
                    "main_category": row[2].strip(),
                }
    return out


def normalize_main_category(text: str) -> str:
    return str(text or "").strip().lower().replace("-", " ")


def target_group(main_category: str) -> str:
    main = normalize_main_category(main_category)
    clothing = {"all body", "bottoms", "tops", "outerwear"}
    accessory = {"bags", "shoes", "accessories", "hats", "jewellery", "scarves", "sunglasses"}
    if main in clothing:
        return "clothing-led"
    if main in accessory:
        return "accessory-led"
    return "other"


def item_title(meta: dict, item_id: str) -> str:
    info = meta.get(str(item_id), {})
    return clean_text(info.get("title") or info.get("url_name") or str(item_id))


def item_category(meta: dict, cat_map: dict, item_id: str) -> tuple:
    info = meta.get(str(item_id), {})
    cid = str(info.get("category_id", ""))
    mapped = cat_map.get(cid, {})
    fine = mapped.get("fine_category", "")
    main = mapped.get("main_category", info.get("semantic_category", ""))
    return cid, fine, main


def item_image_path(item_id: str) -> str:
    return str(POLYVORE_ROOT / "images" / f"{item_id}.jpg")


def load_fragments(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            if obj.get("valid") is not True:
                continue
            frags = obj.get("fragments", {})
            ablation = obj.get("title_ablation", {})
            weather_terms = frags.get("weather", [])
            occasion_terms = frags.get("occasion", [])
            style_terms = frags.get("style", [])
            title_full = clean_text(obj.get("title", ""))
            weather_text = clean_text(weather_terms)
            temp = parse_temperature(weather_text)
            if pd.isna(temp):
                temp = parse_temperature(title_full)
            rows.append({
                "set_id": str(obj.get("id", "")),
                "title_full_text": title_full,
                "weather_terms": weather_terms if isinstance(weather_terms, list) else [],
                "occasion_terms": occasion_terms if isinstance(occasion_terms, list) else [],
                "style_terms": style_terms if isinstance(style_terms, list) else [],
                "weather_text": weather_text,
                "occasion_text": clean_text(occasion_terms),
                "style_text": clean_text(style_terms),
                "style_richness": len([x for x in style_terms if str(x).strip()]) if isinstance(style_terms, list) else 0,
                "title_no_weather_text": clean_text(ablation.get("no_weather", "")),
                "title_no_occasion_text": clean_text(ablation.get("no_occasion", "")),
                "title_no_style_text": clean_text(ablation.get("no_style", "")),
                "temperature_c": temp,
            })
    return pd.DataFrame(rows)


def controlled_counterfactual_text(row: pd.Series, factor: str, direction: str) -> tuple:
    # 只替換對應因子；其餘部分使用既有 title_ablation.no_* 欄位保留。
    if factor == "weather":
        core = clean_text(row.get("title_no_weather_text", ""))
        if direction == "warm_to_cold":
            replacement = "10°C, cool layered weather"
        else:
            replacement = "30°C, warm breathable weather"
        return clean_text(f"{replacement}, {core}"), replacement, core

    if factor == "occasion":
        core = clean_text(row.get("title_no_occasion_text", ""))
        if direction == "formal_to_casual":
            replacement = "for a casual daily outing"
        else:
            replacement = "for a formal work meeting"
        return clean_text(f"{core}, {replacement}"), replacement, core

    if factor == "style":
        core = clean_text(row.get("title_no_style_text", ""))
        if direction == "high_to_low_style":
            replacement = "simple minimal look"
        else:
            replacement = "polished layered elegant style with coordinated details"
        return clean_text(f"{core}, {replacement}"), replacement, core

    raise ValueError(f"Unknown factor/direction: {factor}/{direction}")


print("Notebook paths are ready.")
print("FRAG_JSONL:", FRAG_JSONL)
print("PAIR_CSV:", PAIR_CSV)
print("RETRIEVAL_CSV:", RETRIEVAL_CSV)
print("REVIEW_CSV:", REVIEW_CSV)
print("FIG_SVG_A:", FIG_SVG_A)
print("FIG_SVG_B:", FIG_SVG_B)


## 1. 建立三因子反事實配對樣本

抽樣母體限定為正式 CIR 可評估的 9,311 筆 query-target pairs。三個因素與既有消融實驗一致：

- weather：對應 `no_weather`
- occasion：對應 `no_occasion`
- style：對應 `no_style`

每個方向挑 4 筆，並優先讓目標單品類別分散於 clothing-led 與 accessory-led，避免案例集中在鞋子或配件。


In [2]:
# ------------------------------------------------------------
# 1. 建立 A44：三因子反事實配對樣本
# ------------------------------------------------------------
fragments = load_fragments(FRAG_JSONL)
detail = pd.read_csv(
    DETAIL_CIR_CSV,
    dtype={"set_id": str, "target_item_id": str, "target_item_fg": str},
)
detail["official_context_aware_rank_seed1"] = detail["rank"].astype(int)
detail["official_context_aware_hit10_seed1"] = detail["hit@10"].astype(int)

test_data = load_json(POLYVORE_ROOT / "disjoint" / "test.json")
train_data = load_json(POLYVORE_ROOT / "disjoint" / "train.json")
fitb_data = load_json(POLYVORE_ROOT / "disjoint" / "fill_in_blank_test.json")
meta = load_json(POLYVORE_ROOT / "polyvore_item_metadata.json")
cat_map = read_categories(POLYVORE_ROOT / "categories.csv")

ref_to_item = {}
full_items_by_set = {}
for outfit in test_data:
    sid = str(outfit["set_id"])
    items = []
    for item in outfit["items"]:
        item_id = str(item["item_id"])
        ref_to_item[f"{sid}_{int(item['index'])}"] = item_id
        items.append(item_id)
    full_items_by_set[sid] = items

official_set_ids = set(detail["set_id"])
fitb_rows = []
for q in fitb_data:
    if not q.get("question"):
        continue
    sid = str(q["question"][0]).split("_")[0]
    if sid not in official_set_ids:
        continue
    partial_ids = [ref_to_item[x] for x in q["question"] if x in ref_to_item]
    answer_ids = [ref_to_item[x] for x in q["answers"] if x in ref_to_item]
    correct_refs = [x for x in q["answers"] if str(x).split("_")[0] == sid]
    if not correct_refs:
        continue
    target_item_id = ref_to_item[correct_refs[0]]
    cid, fine, main = item_category(meta, cat_map, target_item_id)
    fitb_rows.append({
        "set_id": sid,
        "blank_position": q.get("blank_position", ""),
        "partial_item_ids": " | ".join(partial_ids),
        "partial_item_titles": " | ".join(item_title(meta, x) for x in partial_ids),
        "fitb_answer_item_ids": " | ".join(answer_ids),
        "target_item_id_from_fitb": target_item_id,
        "target_category_id": cid,
        "target_fine_category": fine,
        "target_main_category": main,
        "target_group": target_group(main),
        "target_item_title": item_title(meta, target_item_id),
        "target_image_path": item_image_path(target_item_id),
        "partial_image_paths": " | ".join(item_image_path(x) for x in partial_ids),
        "full_outfit_item_ids": " | ".join(full_items_by_set.get(sid, [])),
    })
fitb_df = pd.DataFrame(fitb_rows)

work = (
    fragments.merge(
        detail[[
            "set_id", "target_item_id", "target_item_fg",
            "official_context_aware_rank_seed1",
            "official_context_aware_hit10_seed1",
        ]],
        on="set_id",
        how="inner",
    )
    .merge(fitb_df, on="set_id", how="inner")
)
work = work[work["target_item_id"] == work["target_item_id_from_fitb"]].copy()

temperature_threshold = float(work["temperature_c"].dropna().median())
source_for_occasion = (work["occasion_text"].fillna("") + " " + work["title_full_text"].fillna(""))
work["is_formal"] = source_for_occasion.map(lambda x: bool(FORMAL_RE.search(str(x))))
work["is_casual"] = source_for_occasion.map(lambda x: bool(CASUAL_RE.search(str(x))))
work["is_high_style"] = work["style_richness"].astype(int) >= 2
work["is_low_style"] = work["style_richness"].astype(int) <= 1

print(f"official CIR evaluable rows: {len(detail):,}")
print(f"joined rows with fragments and FITB question: {len(work):,}")
print(f"temperature median threshold used in this notebook: {temperature_threshold:.2f}°C")

specs = [
    ("weather", "warm_to_cold", "Weather: Warm", "Weather: Cold", lambda d: d["temperature_c"] > temperature_threshold + 2),
    ("weather", "cold_to_warm", "Weather: Cold", "Weather: Warm", lambda d: d["temperature_c"] <= temperature_threshold - 2),
    ("occasion", "formal_to_casual", "Occasion: Formal", "Occasion: Casual", lambda d: d["is_formal"] & ~d["is_casual"]),
    ("occasion", "casual_to_formal", "Occasion: Casual", "Occasion: Formal", lambda d: d["is_casual"] & ~d["is_formal"]),
    ("style", "high_to_low_style", "Style richness: High", "Style richness: Low", lambda d: d["is_high_style"]),
    ("style", "low_to_high_style", "Style richness: Low", "Style richness: High", lambda d: d["is_low_style"]),
]

rng = np.random.default_rng(RANDOM_SEED)
used_set_ids = set()
selected_rows = []


def pick_balanced(pool: pd.DataFrame, n: int, used: set) -> pd.DataFrame:
    pool = pool[~pool["set_id"].isin(used)].copy()
    if len(pool) < n:
        pool = pool.copy()
    pool["_rand"] = rng.random(len(pool))
    pool = pool.sort_values("_rand")

    picked_indices = []
    used_main = set()

    # 每個方向優先選 2 筆 clothing-led 與 2 筆 accessory-led；
    # 同一方向中也盡量不要重複 target_main_category。
    quotas = [("clothing-led", 2), ("accessory-led", 2)]
    for group_name, quota in quotas:
        sub = pool[pool["target_group"] == group_name]
        for _, r in sub.iterrows():
            if len([i for i in picked_indices if pool.loc[i, "target_group"] == group_name]) >= quota:
                break
            if r["target_main_category"] in used_main:
                continue
            picked_indices.append(r.name)
            used_main.add(r["target_main_category"])

        if len([i for i in picked_indices if pool.loc[i, "target_group"] == group_name]) < quota:
            for _, r in sub.iterrows():
                if r.name in picked_indices:
                    continue
                picked_indices.append(r.name)
                if len([i for i in picked_indices if pool.loc[i, "target_group"] == group_name]) >= quota:
                    break

    if len(picked_indices) < n:
        for _, r in pool.iterrows():
            if r.name not in picked_indices:
                picked_indices.append(r.name)
            if len(picked_indices) >= n:
                break

    if len(picked_indices) < n:
        raise RuntimeError(f"Not enough samples after balanced sampling: {len(picked_indices)} < {n}")
    return pool.loc[picked_indices[:n]].drop(columns=["_rand"]).sort_values(["target_group", "target_main_category", "set_id"])


for factor, direction, base_label, cf_label, filt in specs:
    pool = work[filt(work)].copy()
    if len(pool) < N_PER_DIRECTION:
        raise RuntimeError(f"Not enough rows for {factor}/{direction}: {len(pool)}")

    picked = pick_balanced(pool, N_PER_DIRECTION, used_set_ids)
    for _, row in picked.iterrows():
        pair_id = f"CF{len(selected_rows) + 1:02d}"
        used_set_ids.add(row["set_id"])
        cf_text, replacement, preserved_core = controlled_counterfactual_text(row, factor, direction)
        selected_rows.append({
            "pair_id": pair_id,
            "set_id": row["set_id"],
            "factor": factor,
            "direction": direction,
            "baseline_factor_level": base_label,
            "counterfactual_factor_level": cf_label,
            "context_aware_description": clean_text(row["title_full_text"]),
            "counterfactual_description": cf_text,
            "counterfactual_replacement_terms": replacement,
            "preserved_non_target_factor_core": preserved_core,
            "rewrite_basis": f"title_ablation.no_{factor}",
            "weather_text": row["weather_text"],
            "occasion_text": row["occasion_text"],
            "style_text": row["style_text"],
            "style_richness": int(row["style_richness"]),
            "temperature_c": row["temperature_c"],
            "temperature_threshold_for_subset": temperature_threshold,
            "target_item_id": row["target_item_id"],
            "target_item_title": row["target_item_title"],
            "target_item_fg": row["target_item_fg"],
            "target_category_id": row["target_category_id"],
            "target_fine_category": row["target_fine_category"],
            "target_main_category": row["target_main_category"],
            "target_group": row["target_group"],
            "partial_item_ids": row["partial_item_ids"],
            "partial_item_titles": row["partial_item_titles"],
            "fitb_answer_item_ids": row["fitb_answer_item_ids"],
            "full_outfit_item_ids": row["full_outfit_item_ids"],
            "partial_image_paths": row["partial_image_paths"],
            "target_image_path": row["target_image_path"],
            "official_context_aware_rank_seed1": int(row["official_context_aware_rank_seed1"]),
            "official_context_aware_hit10_seed1": int(row["official_context_aware_hit10_seed1"]),
        })

pairs = pd.DataFrame(selected_rows)
save_csv(PAIR_CSV, pairs)

print(f"[完成] A44 反事實配對樣本：{PAIR_CSV}")
display(pairs.groupby(["factor", "direction"]).size().rename("n").reset_index())
display(pairs.groupby(["factor", "direction", "target_group", "target_main_category"]).size().rename("n").reset_index())
display(pairs[[
    "pair_id", "set_id", "factor", "direction",
    "target_group", "target_main_category", "target_fine_category",
    "baseline_factor_level", "counterfactual_factor_level",
]])


official CIR evaluable rows: 9,311
joined rows with fragments and FITB question: 9,311
temperature median threshold used in this notebook: 22.90°C
[完成] A44 反事實配對樣本：/home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A44_counterfactual_context_pairs.csv


,factor,direction,n
0,occasion,casual_to_formal,4
1,occasion,formal_to_casual,4
2,style,high_to_low_style,4
3,style,low_to_high_style,4
4,weather,cold_to_warm,4
5,weather,warm_to_cold,4


,factor,direction,target_group,target_main_category,n
0,occasion,casual_to_formal,accessory-led,jewellery,1
1,occasion,casual_to_formal,accessory-led,sunglasses,1
2,occasion,casual_to_formal,clothing-led,outerwear,1
3,occasion,casual_to_formal,clothing-led,tops,1
4,occasion,formal_to_casual,accessory-led,jewellery,1
5,occasion,formal_to_casual,accessory-led,shoes,1
6,occasion,formal_to_casual,clothing-led,all-body,1
7,occasion,formal_to_casual,clothing-led,tops,1
8,style,high_to_low_style,accessory-led,bags,1
9,style,high_to_low_style,accessory-led,shoes,1


,pair_id,set_id,factor,direction,target_group,target_main_category,target_fine_category,baseline_factor_level,counterfactual_factor_level
0,CF01,216751861,weather,warm_to_cold,accessory-led,bags,purse,Weather: Warm,Weather: Cold
1,CF02,218410318,weather,warm_to_cold,accessory-led,sunglasses,sunglasses,Weather: Warm,Weather: Cold
2,CF03,214614273,weather,warm_to_cold,clothing-led,outerwear,trench coat,Weather: Warm,Weather: Cold
3,CF04,190756032,weather,warm_to_cold,clothing-led,tops,turtleneck sweater,Weather: Warm,Weather: Cold
4,CF05,133539481,weather,cold_to_warm,accessory-led,bags,clutch,Weather: Cold,Weather: Warm
5,CF06,198740117,weather,cold_to_warm,accessory-led,jewellery,earrings,Weather: Cold,Weather: Warm
6,CF07,31317843,weather,cold_to_warm,clothing-led,all-body,dress,Weather: Cold,Weather: Warm
7,CF08,211329660,weather,cold_to_warm,clothing-led,outerwear,parka,Weather: Cold,Weather: Warm
8,CF09,222693789,occasion,formal_to_casual,accessory-led,jewellery,bracelet,Occasion: Formal,Occasion: Casual
9,CF10,221470979,occasion,formal_to_casual,accessory-led,shoes,heels,Occasion: Formal,Occasion: Casual


## 2. 固定模型與候選池，重新計算排序

使用既有 context-aware CIR 模型 `cir_new_seed1/ckpt.pt`。context-aware baseline 使用既有 FashionCLIP outfit feature；counterfactual 描述重新以 FashionCLIP 編碼。除此之外，partial outfit、target item、target category 與 3000 同類候選池皆保持不變。


In [3]:
# ------------------------------------------------------------
# 2. 建立 A45：反事實前後的排序結果
# ------------------------------------------------------------
if RUN_MODEL:
    import torch
    from fashion_clip.fashion_clip import FashionCLIP

    sys.path.insert(0, str(MODEL_ROOT))
    from dataset import UIUCPolyvoreRetrievalDataset
    from outfit_transformer import OutfitTransformerRetrieval, LinearImageEncoder, LinearTextEncoder
    from config.cir_cond_hardneg import (
        img_inp_size, img_emb_size, txt_inp_size, txt_emb_size,
        outfit_txt_inp_size, nhead, num_layers, margin, max_item_len, use_outfit_txt,
    )

    pairs = pd.read_csv(PAIR_CSV, dtype=str)
    pairs["official_context_aware_rank_seed1"] = pairs["official_context_aware_rank_seed1"].astype(int)

    print("[載入] FashionCLIP：用於 counterfactual description 編碼。")
    fclip = FashionCLIP("fashion-clip")
    cf_vectors = np.asarray(
        fclip.encode_text(pairs["counterfactual_description"].fillna("").tolist(), batch_size=16),
        dtype=np.float32,
    )
    cf_by_pair = {pid: vec for pid, vec in zip(pairs["pair_id"], cf_vectors)}

    with open(FEATURE_ROOT / "encoded_NewoutfitUrlTitle_en_fashionClip.pkl", "rb") as f:
        baseline_outfit_feats = pickle.load(f)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    ctx = nullcontext() if device == "cpu" else torch.amp.autocast(device_type="cuda", dtype=torch.float16)
    print("[裝置]", device)

    common_kwargs = dict(
        datadir=str(VALID_ROOT / "polyvore_data"),
        txt_feat_path=str(FEATURE_ROOT / "encoded_title_description_distiluse-base-multilingual-cased-v2.pkl"),
        img_feat_path=str(FEATURE_ROOT / "img_feats_fashionClip.pkl"),
        category_feat_path=str(FEATURE_ROOT / "encoded_category_distiluse-base-multilingual-cased-v2.pkl"),
        polyvore_split="disjoint",
        max_item_len=max_item_len,
        transform=None,
        sample_hard_negative=False,
    )
    outfit_feat_path = FEATURE_ROOT / "encoded_NewoutfitUrlTitle_en_fashionClip.pkl"
    train_dat = UIUCPolyvoreRetrievalDataset(split="train", outfit_feat_path=str(outfit_feat_path), **common_kwargs)
    test_dat = UIUCPolyvoreRetrievalDataset(split="test", outfit_feat_path=str(outfit_feat_path), **common_kwargs)

    model = OutfitTransformerRetrieval(
        img_encoder=LinearImageEncoder(img_inp_size, img_emb_size),
        text_encoder=LinearTextEncoder(txt_inp_size, txt_emb_size),
        outfit_txt_encoder=LinearTextEncoder(outfit_txt_inp_size, txt_emb_size + img_emb_size),
        nhead=nhead,
        num_layers=num_layers,
        margin=margin,
        target_item_info="category",
        use_outfit_txt=use_outfit_txt,
    )
    ckpt_path = MODEL_ROOT / "experiments" / f"cir_new_seed{SEED}" / "ckpt.pt"
    checkpoint = torch.load(str(ckpt_path), map_location=device)
    model.load_state_dict(checkpoint["model"])
    model.to(device)
    model.eval()
    print("[載入] 模型 checkpoint：", ckpt_path)

    def build_pool_for_fg(fg: str) -> list:
        ids = []
        seen = set()
        for dat in (test_dat, train_dat):
            for _, item_list in dat.fg2ims.get(str(fg), {}).items():
                for item_id in item_list:
                    item_id = str(item_id)
                    if item_id not in seen:
                        seen.add(item_id)
                        ids.append(item_id)
                    if len(ids) >= MIN_OFFICIAL_POOL:
                        return ids[:MIN_OFFICIAL_POOL]
        return ids

    target_fgs = sorted(pairs["target_item_fg"].astype(str).unique())
    candidate_ids = {}
    for fg in target_fgs:
        ids = build_pool_for_fg(fg)
        if len(ids) < MIN_OFFICIAL_POOL:
            raise RuntimeError(f"FG={fg} has only {len(ids)} candidates; official CIR requires >=3000.")
        candidate_ids[fg] = np.array(ids[:MIN_OFFICIAL_POOL], dtype=str)
    print("[完成] 候選池類別數：", len(candidate_ids))

    item_emb_cache = {}

    def extract_item_emb(item_id: str):
        item_id = str(item_id)
        if item_id not in item_emb_cache:
            emb = test_dat.extract_emb({"item_id": item_id}, model, device, ctx, to_numpy=True)
            item_emb_cache[item_id] = np.asarray(emb, dtype=np.float32)
        return item_emb_cache[item_id]

    candidate_feats = {}
    for fg, ids in candidate_ids.items():
        embs = [extract_item_emb(x) for x in ids]
        candidate_feats[fg] = torch.from_numpy(np.concatenate(embs, axis=0)).float()
        print(f"[完成] FG={fg} candidate embeddings: {len(ids)}")

    def stack_if_list(x):
        return torch.stack(x) if isinstance(x, list) else x

    def compute_logits(partial_ids, target_item_id, outfit_vec):
        partial = {"items": [{"item_id": str(x)} for x in partial_ids]}
        partial_imgs, partial_txts, _ = test_dat.load_outfit(partial)
        partial_imgs = stack_if_list(partial_imgs)
        partial_txts = stack_if_list(partial_txts)
        partial_imgs, partial_txts, mask = test_dat.pad_imgs_and_txts(partial_imgs, partial_txts)
        partial_imgs = partial_imgs.to(device)
        partial_txts = partial_txts.to(device)
        mask = mask.to(device)
        target_category = torch.tensor(test_dat.category_feats[str(target_item_id)]).to(device)
        outfit_txt = torch.tensor(np.asarray(outfit_vec, dtype=np.float32)).to(device)
        with torch.no_grad(), ctx:
            out = model(
                partial_imgs.unsqueeze(0),
                partial_txts.unsqueeze(0),
                mask.unsqueeze(0),
                positive_category=target_category.unsqueeze(0),
                outfit_txt=outfit_txt.unsqueeze(0),
            )
        return out["logits"].detach().cpu().float()

    def rank_candidates(logits, fg, target_item_id):
        cur_ids = deepcopy(candidate_ids[str(fg)])
        cur_feats = candidate_feats[str(fg)].clone()
        target_item_id = str(target_item_id)
        if target_item_id not in set(cur_ids.tolist()):
            cur_ids = np.concatenate([cur_ids[:-1], np.array([target_item_id], dtype=str)])
            target_emb = torch.from_numpy(extract_item_emb(target_item_id)).float()
            cur_feats = torch.cat([cur_feats[:-1, :], target_emb], dim=0)
        scores = torch.cosine_similarity(logits, cur_feats)
        values, indices = torch.sort(scores, descending=True)
        sorted_ids = cur_ids[indices.numpy()].astype(str)
        sorted_scores = values.numpy().astype(float)
        rank = int(np.where(sorted_ids == target_item_id)[0][0]) + 1
        return rank, sorted_ids, sorted_scores

    retrieval_rows = []
    for _, row in pairs.iterrows():
        partial_ids = [x.strip() for x in str(row["partial_item_ids"]).split("|") if x.strip()]
        target_item_id = str(row["target_item_id"])
        fg = str(row["target_item_fg"])
        baseline_vec = np.asarray(baseline_outfit_feats[str(row["set_id"])], dtype=np.float32)
        cf_vec = cf_by_pair[row["pair_id"]]

        for condition, query_text, outfit_vec in [
            ("context_aware", row["context_aware_description"], baseline_vec),
            ("counterfactual", row["counterfactual_description"], cf_vec),
        ]:
            logits = compute_logits(partial_ids, target_item_id, outfit_vec)
            rank, sorted_ids, sorted_scores = rank_candidates(logits, fg, target_item_id)
            retrieval_rows.append({
                "pair_id": row["pair_id"],
                "condition": condition,
                "seed": SEED,
                "set_id": row["set_id"],
                "factor": row["factor"],
                "direction": row["direction"],
                "target_group": row["target_group"],
                "target_main_category": row["target_main_category"],
                "baseline_factor_level": row["baseline_factor_level"],
                "counterfactual_factor_level": row["counterfactual_factor_level"],
                "query_text": query_text,
                "target_item_id": target_item_id,
                "target_item_title": row["target_item_title"],
                "target_item_fg": fg,
                "target_fine_category": row["target_fine_category"],
                "rank": rank,
                "hit@1": int(rank <= 1),
                "hit@5": int(rank <= 5),
                "hit@10": int(rank <= 10),
                "top1_id": sorted_ids[0],
                "top5_ids": " | ".join(sorted_ids[:5].tolist()),
                "top10_ids": " | ".join(sorted_ids[:10].tolist()),
                "top10_scores": " | ".join(f"{x:.6f}" for x in sorted_scores[:10]),
                "official_detail_context_aware_rank_seed1": int(row["official_context_aware_rank_seed1"]),
            })
        print("[完成]", row["pair_id"], row["set_id"], row["factor"], row["direction"])

    retrieval = pd.DataFrame(retrieval_rows)
    save_csv(RETRIEVAL_CSV, retrieval)
    print(f"[完成] A45 排序結果：{RETRIEVAL_CSV}")
else:
    retrieval = pd.read_csv(RETRIEVAL_CSV, dtype=str)
    retrieval["rank"] = retrieval["rank"].astype(int)
    print("[略過模型] 讀取既有 A45：", RETRIEVAL_CSV)

check = retrieval[retrieval["condition"] == "context_aware"].copy()
check["rank_diff_vs_detail"] = check["rank"].astype(int) - check["official_detail_context_aware_rank_seed1"].astype(int)
print("[檢查] context-aware 重新排序與 detail_cir_new_seed1.csv 的 rank 差異：")
display(check["rank_diff_vs_detail"].describe())
display(retrieval.head())


2026-08-06 19:02:16.483974: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-06 19:02:16.494411: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1786014136.506625   73093 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786014136.510726   73093 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1786014136.521851   73093 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

[載入] FashionCLIP：用於 counterfactual description 編碼。


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Map:   0%|          | 0/24 [00:00<?, ? examples/s]

  0%|          | 0/1 [00:00<?, ?it/s]

...

[完成] FG=57 candidate embeddings: 3000


[完成] FG=62 candidate embeddings: 3000


[完成] FG=64 candidate embeddings: 3000
[完成] CF01 216751861 weather warm_to_cold
[完成] CF02 218410318 weather warm_to_cold
[完成] CF03 214614273 weather warm_to_cold
[完成] CF04 190756032 weather warm_to_cold
[完成] CF05 133539481 weather cold_to_warm
[完成] CF06 198740117 weather cold_to_warm
[完成] CF07 31317843 weather cold_to_warm
[完成] CF08 211329660 weather cold_to_warm
[完成] CF09 222693789 occasion formal_to_casual
[完成] CF10 221470979 occasion formal_to_casual
[完成] CF11 222409288 occasion formal_to_casual
[完成] CF12 121188291 occasion formal_to_casual
[完成] CF13 146296814 occasion casual_to_formal
[完成] CF14 213811620 occasion casual_to_formal
[完成] CF15 39196989 occasion casual_to_formal
[完成] CF16 215708922 occasion casual_to_formal
[完成] CF17 59207602 style high_to_low_style
[完成] CF18 224072051 style high_to_low_style
[完成] CF19 207497813 style high_to_low_style
[完成] CF20 174085577 style high_to_low_style
[完成] CF21 219326466 style low_to_high_style
[完成] CF22 156067516 style low_to_high_style
[完成] 

count    24.000000
mean      0.083333
std       0.503610
min      -1.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000
Name: rank_diff_vs_detail, dtype: float64

,pair_id,condition,seed,set_id,factor,direction,target_group,target_main_category,baseline_factor_level,counterfactual_factor_level,...,target_fine_category,rank,hit@1,hit@5,hit@10,top1_id,top5_ids,top10_ids,top10_scores,official_detail_context_aware_rank_seed1
0,CF01,context_aware,1,216751861,weather,warm_to_cold,accessory-led,bags,Weather: Warm,Weather: Cold,...,purse,117,0,0,0,192320374,192320374 | 201368819 | 191986240 | 205559078 ...,192320374 | 201368819 | 191986240 | 205559078 ...,0.762265 | 0.692386 | 0.690314 | 0.673694 | 0....,117
1,CF01,counterfactual,1,216751861,weather,warm_to_cold,accessory-led,bags,Weather: Warm,Weather: Cold,...,purse,116,0,0,0,192320374,192320374 | 191986240 | 201368819 | 202107491 ...,192320374 | 191986240 | 201368819 | 202107491 ...,0.684296 | 0.682399 | 0.663542 | 0.635096 | 0....,117
2,CF02,context_aware,1,218410318,weather,warm_to_cold,accessory-led,sunglasses,Weather: Warm,Weather: Cold,...,sunglasses,9,0,0,1,168668833,168668833 | 112978685 | 196545171 | 161283240 ...,168668833 | 112978685 | 196545171 | 161283240 ...,0.635145 | 0.629251 | 0.599430 | 0.593216 | 0....,9
3,CF02,counterfactual,1,218410318,weather,warm_to_cold,accessory-led,sunglasses,Weather: Warm,Weather: Cold,...,sunglasses,8,0,0,1,168668833,168668833 | 112978685 | 161283240 | 196545171 ...,168668833 | 112978685 | 161283240 | 196545171 ...,0.651526 | 0.630213 | 0.603308 | 0.600715 | 0....,9
4,CF03,context_aware,1,214614273,weather,warm_to_cold,clothing-led,outerwear,Weather: Warm,Weather: Cold,...,trench coat,798,0,0,0,195306869,195306869 | 61510900 | 126458988 | 116305755 |...,195306869 | 61510900 | 126458988 | 116305755 |...,0.795233 | 0.757607 | 0.748093 | 0.724549 | 0....,799


## 3. 產生人工覆核表與摘要

人工覆核重點：被修改的因子是否朝合理方向變化、未修改的因素是否維持、視覺內容是否穩定、推薦排序變化是否可解釋。這不是新的主實驗檢定，而是和三因子消融相互對照的反事實質性檢查。


In [4]:
# ------------------------------------------------------------
# 3. 建立 A46：人工覆核表
# ------------------------------------------------------------
pairs = pd.read_csv(PAIR_CSV, dtype=str)
retrieval = pd.read_csv(RETRIEVAL_CSV, dtype=str)
retrieval["rank"] = retrieval["rank"].astype(int)

base = retrieval[retrieval["condition"] == "context_aware"].set_index("pair_id")
cf = retrieval[retrieval["condition"] == "counterfactual"].set_index("pair_id")

review_rows = []
for _, row in pairs.iterrows():
    pid = row["pair_id"]
    b = base.loc[pid]
    c = cf.loc[pid]
    b_top5 = [x.strip() for x in b["top5_ids"].split("|") if x.strip()]
    c_top5 = [x.strip() for x in c["top5_ids"].split("|") if x.strip()]
    overlap = len(set(b_top5) & set(c_top5)) / max(1, len(set(b_top5) | set(c_top5)))
    rank_delta = int(c["rank"]) - int(b["rank"])
    review_rows.append({
        "pair_id": pid,
        "set_id": row["set_id"],
        "factor": row["factor"],
        "direction": row["direction"],
        "target_group": row["target_group"],
        "target_main_category": row["target_main_category"],
        "target_fine_category": row["target_fine_category"],
        "baseline_factor_level": row["baseline_factor_level"],
        "counterfactual_factor_level": row["counterfactual_factor_level"],
        "context_aware_description": row["context_aware_description"],
        "counterfactual_description": row["counterfactual_description"],
        "counterfactual_replacement_terms": row["counterfactual_replacement_terms"],
        "preserved_non_target_factor_core": row["preserved_non_target_factor_core"],
        "rewrite_basis": row["rewrite_basis"],
        "target_item_id": row["target_item_id"],
        "target_item_title": row["target_item_title"],
        "partial_item_titles": row["partial_item_titles"],
        "context_aware_rank": int(b["rank"]),
        "counterfactual_rank": int(c["rank"]),
        "rank_delta_counterfactual_minus_context": rank_delta,
        "context_aware_hit10": int(b["hit@10"]),
        "counterfactual_hit10": int(c["hit@10"]),
        "top1_changed": int(str(b["top1_id"]) != str(c["top1_id"])),
        "top5_jaccard_overlap": round(overlap, 4),
        "context_aware_top5_ids": b["top5_ids"],
        "counterfactual_top5_ids": c["top5_ids"],
        "target_factor_reasonable_change": "",
        "non_target_factors_preserved": "",
        "visual_content_stable": "",
        "retrieval_change_explainable": "",
        "drift_level": "",
        "review_note": "",
    })

review = pd.DataFrame(review_rows)
save_csv(REVIEW_CSV, review)

summary = review.groupby(["factor", "direction", "target_group"]).agg(
    n_pairs=("pair_id", "count"),
    mean_context_rank=("context_aware_rank", "mean"),
    mean_counterfactual_rank=("counterfactual_rank", "mean"),
    mean_rank_delta=("rank_delta_counterfactual_minus_context", "mean"),
    top1_change_rate=("top1_changed", "mean"),
    mean_top5_jaccard=("top5_jaccard_overlap", "mean"),
).reset_index()

print(f"[完成] A46 人工覆核表：{REVIEW_CSV}")
display(summary)
display(review[[
    "pair_id", "factor", "direction", "target_group", "target_main_category",
    "context_aware_rank", "counterfactual_rank",
    "rank_delta_counterfactual_minus_context", "top1_changed", "top5_jaccard_overlap",
]])


[完成] A46 人工覆核表：/home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A46_counterfactual_context_manual_review_sheet.csv


,factor,direction,target_group,n_pairs,mean_context_rank,mean_counterfactual_rank,mean_rank_delta,top1_change_rate,mean_top5_jaccard
0,occasion,casual_to_formal,accessory-led,2,224.5,236.5,12.0,0.5,0.66670
1,occasion,casual_to_formal,clothing-led,2,783.0,722.0,-61.0,0.5,0.11110
2,occasion,formal_to_casual,accessory-led,2,31.5,26.0,-5.5,0.0,0.45835
3,occasion,formal_to_casual,clothing-led,2,90.0,97.0,7.0,0.5,0.42860
4,style,high_to_low_style,accessory-led,2,393.0,599.5,206.5,1.0,0.12500
5,style,high_to_low_style,clothing-led,2,705.5,894.5,189.0,1.0,0.21430
6,style,low_to_high_style,accessory-led,2,77.0,175.5,98.5,1.0,0.05555
7,style,low_to_high_style,clothing-led,2,1325.5,1668.0,342.5,1.0,0.12500
8,weather,cold_to_warm,accessory-led,2,2051.5,2003.5,-48.0,0.0,0.66670
9,weather,cold_to_warm,clothing-led,2,86.0,44.5,-41.5,0.0,0.42860


,pair_id,factor,direction,target_group,target_main_category,context_aware_rank,counterfactual_rank,rank_delta_counterfactual_minus_context,top1_changed,top5_jaccard_overlap
0,CF01,weather,warm_to_cold,accessory-led,bags,117,116,-1,0,0.4286
1,CF02,weather,warm_to_cold,accessory-led,sunglasses,9,8,-1,0,0.6667
2,CF03,weather,warm_to_cold,clothing-led,outerwear,798,629,-169,0,0.6667
3,CF04,weather,warm_to_cold,clothing-led,tops,1606,785,-821,1,0.1111
4,CF05,weather,cold_to_warm,accessory-led,bags,1204,1141,-63,0,0.6667
5,CF06,weather,cold_to_warm,accessory-led,jewellery,2899,2866,-33,0,0.6667
6,CF07,weather,cold_to_warm,clothing-led,all-body,168,81,-87,0,0.4286
7,CF08,weather,cold_to_warm,clothing-led,outerwear,4,8,4,0,0.4286
8,CF09,occasion,formal_to_casual,accessory-led,jewellery,1,1,0,0,0.2500
9,CF10,occasion,formal_to_casual,accessory-led,shoes,62,51,-11,0,0.6667


## 4. 產生代表性 SVG 與實驗說明

主文不宜把兩個案例上下塞在同一張圖中，容易因縮放造成文字與圖片重疊。因此本節輸出兩張獨立 SVG：F34a 為排序反應較明顯案例，F34b 為反應較有限案例。完整 24 筆與人工覆核欄位保留在 A46。


In [ ]:
# ------------------------------------------------------------
# 4. 建立 F34a/F34b：兩張代表案例 SVG；建立 A47：實驗說明
# ------------------------------------------------------------
review = pd.read_csv(REVIEW_CSV, dtype=str)
for col in ["context_aware_rank", "counterfactual_rank", "rank_delta_counterfactual_minus_context", "top1_changed", "top5_jaccard_overlap"]:
    review[col] = pd.to_numeric(review[col])

pairs = pd.read_csv(PAIR_CSV, dtype=str).set_index("pair_id")
retrieval = pd.read_csv(RETRIEVAL_CSV, dtype=str)
ret_by = {(r["pair_id"], r["condition"]): r for _, r in retrieval.iterrows()}

def data_uri(item_id: str) -> str:
    path = POLYVORE_ROOT / "images" / f"{item_id}.jpg"
    data = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:image/jpeg;base64,{data}"


def wrap_svg_text(text: str, max_chars: int = 72, max_lines: int = 3) -> list:
    lines = textwrap.wrap(clean_text(text), width=max_chars)
    if len(lines) > max_lines:
        lines = lines[:max_lines]
        lines[-1] = lines[-1].rstrip(".") + "..."
    return lines


def image_box(item_id, x, y, w=88, h=88, stroke=BLUE, sw=3, label=""):
    label_text = f'<text x="{x+w/2}" y="{y+h+18}" text-anchor="middle" class="small">{html.escape(label)}</text>' if label else ""
    return "\n".join([
        f'<rect x="{x}" y="{y}" width="{w}" height="{h}" fill="white" stroke="{stroke}" stroke-width="{sw}"/>',
        f'<image href="{data_uri(item_id)}" x="{x+6}" y="{y+6}" width="{w-12}" height="{h-12}" preserveAspectRatio="xMidYMid meet"/>',
        label_text,
    ])


def image_row(item_ids, x0, y, stroke, labels=None):
    parts = []
    for i, item_id in enumerate(item_ids):
        label = labels[i] if labels and i < len(labels) else f"Top-{i+1}"
        parts.append(image_box(item_id, x0 + i * 104, y, stroke=stroke, label=label))
    return "\n".join(parts)


def text_lines(lines, x, y, klass="note", color=None, line_h=18):
    color_attr = f' style="fill:{color};"' if color else ""
    out = []
    for i, line in enumerate(lines):
        out.append(f'<text x="{x}" y="{y + i * line_h}" class="{klass}"{color_attr}>{html.escape(line)}</text>')
    return "\n".join(out)


def build_single_svg(row, panel_label):
    pid = row["pair_id"]
    pair = pairs.loc[pid]
    b = ret_by[(pid, "context_aware")]
    c = ret_by[(pid, "counterfactual")]

    partial_ids = [x.strip() for x in pair["partial_item_ids"].split("|") if x.strip()]
    target_id = pair["target_item_id"]
    b_top5 = [x.strip() for x in b["top5_ids"].split("|") if x.strip()][:5]
    c_top5 = [x.strip() for x in c["top5_ids"].split("|") if x.strip()][:5]

    response = "larger retrieval response" if panel_label == "A" else "limited retrieval response"
    header = f"Example {panel_label}: {pair['factor']} / {pair['direction']} ({response})"
    meta_line = f"{pair['target_group']}; target={pair['target_main_category']} / {pair['target_fine_category']}"
    rank_line = f"context-aware rank #{int(row['context_aware_rank'])}; counterfactual rank #{int(row['counterfactual_rank'])}; top-5 overlap {row['top5_jaccard_overlap']:.2f}"
    desc_a = wrap_svg_text(pair["context_aware_description"], 82, 2)
    desc_b = wrap_svg_text(pair["counterfactual_description"], 82, 2)

    width = 1240
    height = 620
    parts = [
        '<?xml version="1.0" encoding="UTF-8"?>',
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<style>',
        'text { font-family: Arial, Helvetica, sans-serif; fill: #222; }',
        '.panel { font-size: 22px; font-weight: 700; }',
        '.label { font-size: 18px; font-weight: 700; }',
        '.note { font-size: 15px; }',
        '.small { font-size: 12px; fill: #333; }',
        '</style>',
        '<rect width="100%" height="100%" fill="white"/>',
        f'<text x="24" y="34" class="panel">{html.escape(header)}</text>',
        f'<text x="24" y="58" class="note">{html.escape(meta_line)}</text>',
        f'<text x="24" y="78" class="note">{html.escape(rank_line)}</text>',
        '<text x="24" y="132" class="label">Query outfit</text>',
        image_row(partial_ids, 230, 92, BLUE, labels=[f"Item {i+1}" for i in range(len(partial_ids))]),
        image_box(target_id, 230 + len(partial_ids) * 104 + 18, 92, stroke=RED, sw=4, label="Target"),
        f'<text x="24" y="268" class="label" style="fill:{BLUE};">Context-aware</text>',
        text_lines(desc_a, 230, 246, "note", BLUE, 18),
        image_row(b_top5, 230, 292, BLUE),
        f'<text x="24" y="458" class="label" style="fill:{ORANGE};">Counterfactual</text>',
        text_lines(desc_b, 230, 436, "note", ORANGE, 18),
        image_row(c_top5, 230, 482, ORANGE),
        '</svg>',
    ]
    return "\n".join(parts)


review["abs_rank_delta"] = review["rank_delta_counterfactual_minus_context"].abs()
large_case = review.sort_values(["abs_rank_delta", "top1_changed"], ascending=[False, False]).iloc[0]
small_pool = review.sort_values(["abs_rank_delta", "top5_jaccard_overlap"], ascending=[True, False])
small_case = small_pool[small_pool["pair_id"] != large_case["pair_id"]].iloc[0]

svg_a = build_single_svg(large_case, "A")
svg_b = build_single_svg(small_case, "B")
FIG_SVG_A.write_text(svg_a, encoding="utf-8")
FIG_SVG_B.write_text(svg_b, encoding="utf-8")

# 移除舊的上下合併版，避免後續排版時誤用。
old_combined = FIG_DIR / "F34_counterfactual_context_two_case_preview.svg"
if old_combined.exists():
    old_combined.unlink()

print(f"[完成] F34a SVG：{FIG_SVG_A}")
print(f"[完成] F34b SVG：{FIG_SVG_B}")

note_lines = [
    "Counterfactual context consistency check",
    "",
    f"Sampling frame: official PO-D CIR evaluable query-target pairs, n={pd.read_csv(DETAIL_CIR_CSV).shape[0]}.",
    f"Selected pairs: {len(review)} pairs = 3 factors x 2 directions x {N_PER_DIRECTION} cases.",
    "Factors: weather, occasion, and style, aligned with existing no_weather/no_occasion/no_style ablation experiments.",
    "Model: existing context-aware Hybrid Attention / Outfit Transformer CIR model, seed 1.",
    "Controlled variable: same set_id, same partial outfit, same target item, same target category, and same 3000-item same-category candidate pool.",
    "Changed variable: one textual factor only. The non-target core is taken from the corresponding title_ablation.no_* field.",
    "Sampling note: each direction prioritizes both clothing-led and accessory-led target categories to avoid examples concentrated in shoes or accessories.",
    "Figure note: representative examples are separated into F34a and F34b to avoid vertical compression in manuscript layout.",
    "Interpretation: use A46 for manual review. This experiment is a qualitative robustness/sanity check connected to the three-factor ablation, not a new main performance benchmark.",
    "",
    "Recommended manuscript use:",
    "1. Main text: describe the controlled replacement design and summarize manual review results in one compact table.",
    "2. Figure: include F34a and/or F34b only if representative visual examples are needed.",
    "3. Appendix: place A44-A46 for traceability.",
]
NOTE_TXT.write_text("\n".join(note_lines) + "\n", encoding="utf-8")
print(f"[完成] A47 說明檔：{NOTE_TXT}")
print(NOTE_TXT.read_text(encoding="utf-8"))

print("\n[輸出檔案檢查]")
for path in [PAIR_CSV, RETRIEVAL_CSV, REVIEW_CSV, NOTE_TXT, FIG_SVG_A, FIG_SVG_B]:
    print(path.name, path.exists(), path.stat().st_size if path.exists() else 0)
